In [ ]:
def score_subject(sub, model, test_cand_T, test_cand_ids):
    """Return S (n_test, 102) in load_test(sub) order."""
    _, _, Y_tst = data[sub]
    si = SUB_INDEX[sub]
    Xz = np.stack([znorm_trial(x) for x in Y_tst])
    X = torch.from_numpy(Xz.astype(np.float32)).to(DEVICE)
    S_batches = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(X), 64):
            s_idx = torch.full((len(X[i:i+64]),), si, dtype=torch.long, device=DEVICE)
            z = model(X[i:i+64], s_idx)
            S_batches.append((z @ test_cand_T.T).cpu().numpy())
    S = np.concatenate(S_batches, 0)
    # reorder columns so column j = TEST_IDS[j]
    order = np.array([int(np.where(test_cand_ids == v)[0][0]) for v in TEST_IDS])
    return S[:, order].astype(np.float32)

model = EEGEncoder().to(DEVICE)
model.load_state_dict(torch.load(f'{WEIGHTS_DIR}/decoder.pt', map_location=DEVICE))

test_cand_ids = np.array(sorted(targets.keys()))                  # all videos (train+test)
# restrict scoring to the 102 test videos only
cand_ids = TEST_IDS
cand_T   = torch.stack([torch.from_numpy(targets[int(v)]) for v in cand_ids]).to(DEVICE)

for sub in SUBS:
    S = score_subject(sub, model, cand_T, cand_ids)
    np.savez_compressed(f'{SCORES_DIR}/decoder_sub-{sub:02d}.npz',
                        S=S, test_ids=TEST_IDS.astype(np.int64))
    if sub in DEV_SUBS:
        sc = topk(ranked_ids(S), load_test_labels(sub))
        print(f'sub-{sub:02d}', sc)
    else:
        print(f'sub-{sub:02d} saved (held-out)')

## Score the 102 test videos per subject

For each subject, encode its 102 test EEG recordings → `z` on the hypersphere, then take cosine similarity against the frozen target embedding of each candidate video. Save `scores/decoder_sub-XX.npz`.

In [ ]:
def cv_train(pairs, train_vids, n_folds=5, epochs=60, bs=128, lr=3e-3, verbose=True):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=0)
    scores = []
    best_global = -1
    best_model  = None
    for fi, (tr_v_idx, va_v_idx) in enumerate(kf.split(train_vids)):
        tr_v = set(train_vids[i] for i in tr_v_idx)
        va_v = set(train_vids[i] for i in va_v_idx)
        tr_pairs = [p for p in pairs if p[2] in tr_v]
        va_pairs = [p for p in pairs if p[2] in va_v]
        if verbose:
            print(f'fold {fi+1}/{n_folds}  train trials={len(tr_pairs)} val trials={len(va_pairs)}')
        model, best = train_fold(tr_pairs, va_pairs, epochs=epochs, bs=bs, lr=lr,
                                 verbose=False)
        scores.append(best)
        if best > best_global:
            best_global = best
            best_model = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose:
            print(f'  fold best top1={best:.3f}')
    print('mean fold top1:', np.mean(scores), '+/-', np.std(scores))
    return best_model, scores

best_state, cv_scores = cv_train(pairs, train_vids, n_folds=5, epochs=60, bs=128)
torch.save(best_state, f'{WEIGHTS_DIR}/decoder.pt')
print('saved', f'{WEIGHTS_DIR}/decoder.pt')

In [ ]:
def info_nce(z_eeg, z_vid, log_tau):
    """Symmetric InfoNCE across the batch. All inputs L2-normalised."""
    tau = log_tau.exp().clamp(1e-2, 100.)
    logits = tau * z_eeg @ z_vid.T
    tgt = torch.arange(logits.shape[0], device=logits.device)
    return 0.5 * (F.cross_entropy(logits, tgt) + F.cross_entropy(logits.T, tgt))

def video_identification_topk(model, loader, cand_ids, cand_T, device):
    """Average video id top-1 over the loader. cand_T: (n_vid, D) tensor on device."""
    model.eval()
    with torch.no_grad():
        hits1, hits5, total = 0, 0, 0
        for x, _, s in loader:
            x = x.to(device); s = s.to(device)
            z = model(x, s)                           # (B, D)
            logits = z @ cand_T.T                     # (B, n_vid)
            ranks = (-logits).argsort(1)
            true_vids = torch.tensor([int(loader.dataset.pairs[i][2]) for i
                                      in range(total, total + len(x))], device=device)
            true_idx = torch.tensor([int(np.where(cand_ids == v.item())[0][0])
                                     for v in true_vids], device=device)
            r = (ranks == true_idx[:, None]).int().argmax(1) + 1
            hits1 += (r <= 1).sum().item()
            hits5 += (r <= 5).sum().item()
            total += len(x)
    model.train()
    return hits1 / total, hits5 / total

def train_fold(train_pairs, val_pairs, epochs=60, bs=128, lr=3e-3, wd=1e-4,
               patience=10, verbose=True):
    tr_ds = EEGClipDataset(train_pairs, augment=True)
    va_ds = EEGClipDataset(val_pairs,   augment=False)
    tr = DataLoader(tr_ds, batch_size=bs, shuffle=True,  num_workers=0, drop_last=True)
    va = DataLoader(va_ds, batch_size=bs, shuffle=False, num_workers=0)

    model = EEGEncoder().to(DEVICE)
    log_tau = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=DEVICE))
    opt = torch.optim.AdamW(list(model.parameters()) + [log_tau], lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    val_vids = sorted({int(p[2]) for p in val_pairs})
    val_cand_T = torch.stack([torch.from_numpy(targets[v]) for v in val_vids]).to(DEVICE)
    val_cand_ids = np.array(val_vids)

    best = -1; best_state = None; bad = 0
    for ep in range(epochs):
        model.train()
        tot, n = 0.0, 0
        for x, t, s in tr:
            x, t, s = x.to(DEVICE), t.to(DEVICE), s.to(DEVICE)
            z = model(x, s)
            loss = info_nce(z, t, log_tau)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item() * len(x); n += len(x)
        sched.step()
        top1, top5 = video_identification_topk(model, va, val_cand_ids, val_cand_T, DEVICE)
        if verbose:
            print(f'  ep{ep:03d} loss={tot/n:.3f} val top1={top1:.3f} top5={top5:.3f}')
        if top1 > best:
            best = top1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                if verbose: print(f'  early stop at ep{ep}')
                break
    model.load_state_dict(best_state)
    return model, best

## Training

Symmetric InfoNCE (CLIP-style) with a learnable temperature. 5-fold CV over train videos; the best model (by held-out video-identification top-1 across all 6 subjects) is kept.

In [ ]:
SUB_INDEX = {s: i for i, s in enumerate(SUBS)}        # 1..6 -> 0..5

def build_pairs(data):
    pairs = []
    all_vids = set()
    for s, (ids, Y_trn, _) in data.items():
        si = SUB_INDEX[s]
        for v, eeg in zip(ids, Y_trn):
            pairs.append((si, eeg, int(v)))
            all_vids.add(int(v))
    return pairs, sorted(all_vids)

pairs, train_vids = build_pairs(data)
print('pairs:', len(pairs), 'unique train videos:', len(train_vids))

## Build training pairs

One `(sub_idx, eeg, vid_id)` triple per training trial, pooled across all 6 subjects. For each subject we also keep an index of its train videos so we can CV-split on *videos*, not trials — otherwise the same video (shared across subjects) would appear in both splits.

In [ ]:
def znorm_trial(x, eps=1e-6):
    mu = x.mean(axis=-1, keepdims=True)
    sd = x.std(axis=-1, keepdims=True) + eps
    return (x - mu) / sd

class EEGClipDataset(Dataset):
    def __init__(self, pairs, augment=False, noise_sd=0.05, time_shift=5):
        # pairs: list of (sub_idx, eeg[ch,t], vid_id)
        self.pairs = pairs
        self.aug = augment
        self.noise_sd = noise_sd
        self.time_shift = time_shift

    def __len__(self): return len(self.pairs)

    def __getitem__(self, i):
        s, x, vid = self.pairs[i]
        x = znorm_trial(x)
        if self.aug:
            if self.time_shift:
                k = np.random.randint(-self.time_shift, self.time_shift + 1)
                x = np.roll(x, k, axis=-1)
            if self.noise_sd:
                x = x + self.noise_sd * np.random.randn(*x.shape).astype(np.float32)
        t = targets[int(vid)]
        return (torch.from_numpy(x.astype(np.float32)),
                torch.from_numpy(t),
                torch.tensor(s, dtype=torch.long))

class EEGEncoder(nn.Module):
    def __init__(self, n_sub=6, n_chan=N_CHAN, n_time=N_TIME,
                 n_filt_t=40, n_filt_s=40, kernel_t=25, pool=15, drop=0.3,
                 target_dim=TARGET_DIM):
        super().__init__()
        # subject-specific 1x1 channel remix (lives on CPU/GPU as a ModuleList)
        self.adapter = nn.ModuleList([nn.Conv1d(n_chan, n_chan, 1) for _ in range(n_sub)])
        # ShallowConvNet-like trunk
        self.temporal = nn.Conv2d(1, n_filt_t, (1, kernel_t), bias=False)
        self.spatial  = nn.Conv2d(n_filt_t, n_filt_s, (n_chan, 1), bias=False)
        self.bn       = nn.BatchNorm2d(n_filt_s)
        self.drop     = nn.Dropout(drop)
        self.pool_k   = pool
        with torch.no_grad():
            feat = self._trunk(torch.zeros(1, n_chan, n_time))
        self.proj = nn.Linear(feat.shape[-1], target_dim)

    def _trunk(self, x):
        x = x.unsqueeze(1)                                   # (B, 1, C, T)
        x = self.temporal(x)                                 # (B, F, C, T')
        x = self.spatial(x)                                  # (B, F, 1, T')
        x = self.bn(x).squeeze(2)                            # (B, F, T')
        x = x ** 2
        x = F.avg_pool1d(x, self.pool_k, stride=self.pool_k) # (B, F, T'')
        x = torch.log(x.clamp_min(1e-6))
        x = self.drop(x)
        return x.flatten(1)

    def _apply_adapter(self, x, sub_idx):
        """Route each sample through its subject-specific 1x1 conv."""
        out = torch.empty_like(x)
        for s in torch.unique(sub_idx).tolist():
            m = sub_idx == s
            out[m] = self.adapter[int(s)](x[m])
        return out

    def forward(self, x, sub_idx):
        x = self._apply_adapter(x, sub_idx)
        z = self.proj(self._trunk(x))
        return F.normalize(z, dim=-1)

## Dataset and model

- `EEGClipDataset` yields `(x, target, sub_idx)` samples. `x` is per-trial z-scored; a mild noise augmentation is applied on train only.
- `EEGEncoder` has one 1×1 input conv per subject (channel remix 17→D) and a shared ShallowConvNet-ish trunk (temporal conv, spatial depthwise conv, pooled → projection). Output is L2-normalised to live on the target hypersphere.

In [ ]:
import mne

def bp_decimate(x, l=0.1, h=40., dec=DEC):
    flat = x.reshape(-1, x.shape[-1]).astype(np.float64)
    y = mne.filter.filter_data(flat, sfreq=SFREQ, l_freq=l, h_freq=h, verbose=False)
    y = y.reshape(x.shape)
    y = decimate(y, dec, axis=-1, ftype='fir', zero_phase=True)
    return y.astype(np.float32)

def load_subject(sub, cache=True):
    cp = f'./cache_05/sub-{sub:02d}.npz'
    os.makedirs('./cache_05', exist_ok=True)
    if cache and os.path.exists(cp):
        z = np.load(cp)
        return z['ids'], z['Y_trn'], z['Y_tst']
    ids, Y_trn = load_train(sub)
    Y_tst      = load_test(sub)
    Y_trn, Y_tst = bp_decimate(Y_trn), bp_decimate(Y_tst)
    if cache:
        np.savez_compressed(cp, ids=ids, Y_trn=Y_trn, Y_tst=Y_tst)
    return ids, Y_trn, Y_tst

data = {s: load_subject(s) for s in SUBS}
for s in SUBS:
    ids, Y_trn, Y_tst = data[s]
    print(f'sub-{s:02d}  trn {Y_trn.shape}  tst {Y_tst.shape}')

## EEG preprocessing

Minimal variant for the decoder: band-pass 0.1–40 Hz + decimate ×5 → 100 Hz. We skip MVNN here — the input adapter already learns a subject-specific channel mix. Per-trial z-score is applied on the fly in the dataset.

In [ ]:
TARGET_BANKS = ['clip_vitl14', 'dinov2_vitl14', 'clap']

def _l2(x, axis=-1, eps=1e-6):
    n = np.linalg.norm(x, axis=axis, keepdims=True)
    return x / np.maximum(n, eps)

def build_targets():
    """Return dict {vid:int -> np.ndarray[D]} and the concatenated dim."""
    stacks = []
    for b in TARGET_BANKS:
        z = np.load(f'{FEAT_DIR}/{b}.npz')
        ids = sorted(int(k) for k in z.files)
        X = np.stack([z[f'{v:04d}'] for v in ids]).astype(np.float32)
        X = _l2(X)
        stacks.append((ids, X))
    base_ids = stacks[0][0]
    for ids, _ in stacks[1:]:
        assert ids == base_ids, 'target bank video id lists disagree'
    T = np.concatenate([X for _, X in stacks], axis=1)
    T = _l2(T)
    return {v: T[i] for i, v in enumerate(base_ids)}, T.shape[1]

targets, TARGET_DIM = build_targets()
print('target dim:', TARGET_DIM, 'videos:', len(targets))

## Target embedding per video

Concat the frozen CLIP, DINOv2 and CLAP features; L2-normalise each block, concatenate, then re-normalise. The three sub-blocks each contribute roughly equally to the InfoNCE loss.

In [ ]:
TRAIN_DIR     = './trn'
TEST_DIR      = './tst'
TEST_STIM_DIR = './tst_stim'
FEAT_DIR      = './features'
WEIGHTS_DIR   = './weights'
SCORES_DIR    = './scores'
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(SCORES_DIR,  exist_ok=True)

SFREQ         = 500
DEC           = 5                        # -> 100 Hz, 300 samples / trial
SUBS          = [1, 2, 3, 4, 5, 6]
DEV_SUBS      = [1, 2, 3, 4]
HELDOUT_SUBS  = [5, 6]
N_CHAN        = 17
N_TIME        = 300
TEST_IDS      = np.arange(1001, 1103)

def load_train(sub):
    z = np.load(f'{TRAIN_DIR}/sub-{sub:02d}_trn.npz')
    ids = np.array(sorted(z.files, key=int), dtype=int)
    eeg = np.stack([z[str(i)] for i in ids])
    return ids, eeg

def load_test(sub):
    return np.load(f'{TEST_DIR}/sub-{sub:02d}_tst.npy')

def load_test_labels(sub):
    assert sub in DEV_SUBS
    return np.load(f'{TEST_STIM_DIR}/sub-{sub:02d}_tst_stim.npy').astype(int)

def ranked_ids(S, video_ids=TEST_IDS):
    return np.asarray(video_ids)[np.argsort(-S, axis=1)]

def topk(ranked, truth, ks=(1, 5)):
    hits = ranked == np.asarray(truth)[:, None]
    assert hits.any(1).all()
    rank = hits.argmax(1) + 1
    out = {f'top{k}': float((rank <= k).mean()) for k in ks}
    out['median_rank'] = float(np.median(rank))
    return out

## Config and helpers (duplicated from 00 so this notebook runs standalone)

In [ ]:
import os, glob, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from scipy.signal import decimate
from sklearn.model_selection import KFold

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
torch.manual_seed(0); np.random.seed(0); random.seed(0)

In [ ]:
# Local:  %pip install -r requirements.txt
# Kaggle: features/*.npz must already be mounted (from the krabiki-features dataset)
#         %pip install -q mne  (only extra; torch is preinstalled)

# 05 — Contrastive EEG decoder (GPU)

Train a cross-subject EEG → video-embedding decoder with InfoNCE.

- **Target space**: concat of CLIP ViT-L/14 + DINOv2 ViT-L/14 + CLAP (each L2-normalised, then concatenated and re-normalised). Frozen; comes from `features/*.npz` written by 03.
- **EEG encoder**: subject-specific 1×1 input adapter → shared ShallowConvNet-style trunk → projection to target dim.
- **Training**: all 6 subjects' train data pooled (subject index routes the adapter). 5-fold CV over train videos drives early stopping. Train split uses the 695 videos shared by all subjects when available (plan note), falling back to the subject's full train set otherwise.
- **Rule**: `tst` labels of sub-01..04 are **only** used to score the final matrix. Nothing in training touches them.

Outputs:
- `weights/decoder.pt` — single checkpoint (shared trunk + 6 adapters + projection).
- `scores/decoder_sub-01..06.npz` with `S` of shape `(102, 102)` in `load_test(sub)` order, column `j` = `TEST_IDS[j]`. Consumed by 06.